In [ ]:
import torch
from transformers import pipeline

# Initialize the pipeline for text generation
pipe = pipeline(
    "text-generation",
    model="Qwen/Qwen3.6-27B",
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

config.json:   0%|          | 0.00/4.31k [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/112k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 12.8MB            

tokenizer.json: downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/7.76k [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

In [ ]:
import json

# Illustrations use throwaway names (Sam/Alex) that appear in NO puzzle, so the
# fixed prompt can never prime the model on an actual gold block.
SYSTEM_PROMPT = """Knights and Knaves. Each character makes one statement.
Characters with the role "<<TRUTH_ROLE>>" always speak the truth. Characters with the role "<<LIAR_ROLE>>" always lie.

You are given the names of the characters and one block of someone's reasoning about the puzzle.
You are not given the puzzle's statements. Use only the text of this block.
Do not try to solve the puzzle. Never check whether the block is right.

For EACH character listed, answer one question:

Does the block do BOTH of these for that character?
  (A) say that THAT character's OWN statement is true or false
  (B) connect that truth claim to something -- derive it from something the block states,
      or draw something from it

About (A)
- Any plain wording that says the statement is true or false counts, for example "accurate",
  "mistaken", "honest", "dishonest", and negated forms such as "is not a lie".
- Saying the statement itself is contradictory, or would contradict itself, counts as
  calling it false, for example "Alex's statement is contradictory" or "what Alex said
  would contradict itself". A contradiction said of a situation is NOT a truth value for
  any statement, for example "that leads to a contradiction" or "this cannot happen".
- A truth value inside an assumption counts, for example "assume what he said is accurate".
- It must be about the statement as a unit. If the block itself says the truth value is for
  only one part of the statement, it does not count.
- It must be about that character's OWN statement.
  In "Because Sam is a saint, what Alex said must be false", Sam gets a ROLE and ALEX gets a
  truth value. Only Alex has (A). Sam does not.
- The character may be pointed to by name, by a pronoun whose owner is clear in the block,
  or as part of a group whose members are clear in the block. If the block does not make
  clear whose statement it means, nobody has (A).
- Go by what the block says about who said what. Do not correct it.
- Naming or repeating what a character said, without a truth value for it, is not (A).

About (B)
- What the truth claim is connected to does not matter. It may be that character's own role,
  another character's role, or any other conclusion. The character's own role does NOT have
  to appear.
- Connectives = so, therefore, thus, hence, because, since, means, makes, which, that, if,
  suppose, then. A dash or a comma also counts when one part is given as the reason for the other.
- NOT connectives = separately, also, meanwhile, anyway.
- The connective must link the truth claim itself to something. A connective somewhere else
  in the block does not count.
- Claims that are only listed side by side, with commas or "and", are not connected.
- A truth claim with nothing connected before or after it gives false.
- A truth claim that comes only from that character's own role, as in "Sam is a saint, so his
  statement is true", is not connected when nothing follows from it. If the block draws
  anything from it, even inside an assumption, as in "If Sam is a saint, he tells the truth,
  so Alex is a devil", it is connected.ough.

Wrong reasoning still gives true. A correct answer with no truth claim still gives false.

Reply with one JSON object: one key per listed character, using the names exactly as listed,
value true or false. Nothing else.

The examples below use the roles saint and devil. Your puzzle may use other role names.

Example 1
Characters: Sam, Alex, Jordan
Block: We already found that Sam is a saint. That makes what Alex said true, and only a saint says true things, so Alex is a saint.
{"Sam": false, "Alex": true, "Jordan": false}
Sam has a role but no truth value about what Sam said.

Example 2
Characters: Sam, Alex, Jordan
Block: Since Jordan's claim is mistaken, neither Alex nor Sam can be a devil.
{"Sam": false, "Alex": false, "Jordan": true}
Jordan is true even though the block never gives Jordan's role.

Example 3
Characters: Sam, Alex, Jordan
Block: Alex talks about Sam. Sam is a devil. Meanwhile, Jordan is being honest.
{"Sam": false, "Alex": false, "Jordan": false}
Jordan has a truth value, but nothing is connected to it.

Example 4
Characters: Sam, Alex, Jordan
Block: Since Sam is a saint, his statement is true. Alex's statement is false, so Alex is a devil.
{"Sam": false, "Alex": true, "Jordan": false}
Sam's truth value only repeats the rule for saints, and nothing follows from it."""


USER_TEMPLATE = """Characters: {names}

Block:
{block}"""


def render_user(names, block):
    lines = ", ".join(names)
    return USER_TEMPLATE.format(names=lines, block=block)


def build_messages(characters, block):
    """Chat-format messages for transformers (system goes inside the list)."""
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": render_user(characters, block)},
    ]


# if __name__ == "__main__":
#     # show a rendered prompt for one mixed-label block
#     st = {"Ethan": "Ethan and Mia are both saints.", "Mia": "Ethan is a devil.",
#           "Noah": "Ethan and Mia are both devils."}
#     blk = "Noah's statement - actually, let's start with Mia. Mia is a saint, so her statement is true, so Ethan is a devil."
#     sysp, msgs = build_messages(st, blk, ["Noah", "Mia"])
#     print("=== USER MESSAGE ===")
#     print(msgs[0]["content"])

#     chat_messages = [
#         {"role": "system", "content": sysp},
#         {"role": "user", "content": msgs[0]["content"]}
#     ]
#     chat_prompt = pipe.tokenizer.apply_chat_template(chat_messages, tokenize=False, add_generation_prompt=True)
#     chat_outputs = pipe(chat_prompt, max_new_tokens=512, do_sample=True, temperature=0.1)

#     print("Qwen's Response:")
#     print(chat_outputs[0]["generated_text"][len(chat_prompt):])

    # print("\n=== parse_output self-test ===")
    # fake = '{"Noah": {"link_phrase": null, "engaged": false}, "Mia": {"link_phrase": "Mia is a saint, so her statement is true", "engaged": true}}'
    # print(parse_output(fake, ["Noah", "Mia"]))
    # for bad in ['{"Noah": {"engaged": "yes"}}', '{"Noah": {"link_phrase": null, "engaged": true}}']:
    #     try: parse_output(bad, ["Noah"])
    #     except ValueError as e: print("rejected:", e)


In [ ]:
import json

with open('/content/engaged_tests_all.json', 'r') as f:
  # Use json.load() to read directly from the file object
  file = json.load(f)

print(file)

{'version': 'all three engagement test sets in one file to run, 2 Oct. Labels are the 2 Oct versions, checked against the Sep 22, Sep 26 and Sep 30 rules.', 'how_to_run': "Same notebook and prompt as the 1,186-step run. names = list(puzzle['statements'].keys()), roles from puzzle['roles'], blk = block['text']. The 9 stress_v3 blocks are already done: copy out_engaged_stress_v4.json to a new name and use that as OUT_PATH, so they are skipped and only the other 73 blocks run.", 'how_to_score': "Score only the speakers listed in each block's labels. Report each set separately (field 'set' on every puzzle and block): gold, stress_merged, stress_v3. Never pool them into one number. Report boundary labels (ratify: true) separately and keep them out of the headline numbers.", 'prompt_overlap': "Blocks sharing wording or shape with the prompt's examples: gold pos-02, pos-06, bnd-03, mt-01 (Ethan), dng-01 (Mia), mix-01 (Mia); stress_merged J-04, M-03, D-02 (Mia). Report them but do not count th

In [ ]:
def get_all_keys(obj):
    keys = set()
    if isinstance(obj, dict):
        for k, v in obj.items():
            keys.add(k)
            keys.update(get_all_keys(v))
    elif isinstance(obj, list):
        for item in obj:
            keys.update(get_all_keys(item))
    return keys

all_keys = get_all_keys(file)
print("All unique keys in the JSON structure:")
print(all_keys)


All unique keys in the JSON structure:
{'speaker', 'difficulty', 'how_to_run', 'Mia', 'counts', 'stress_v3', 'roles', 'liar', 'statements', 'Lena', 'note', 'Priya', 'verify', 'puzzles', 'examples', 'Luna', 'Yusuf', 'Noah', 'reason', 'gold', 'truth', 'source', 'Kiran', 'version', 'blocks', 'boundary_labels', 'Ravi', 'blocks_per_set', 'Ella', 'Wen', 'text', 'engaged', 'id', 'how_to_score', 'ratify_note', 'Sara', 'Dev', 'true_labels_excluding_boundary', 'false_labels_excluding_boundary', 'Joseph', 'same_logical_shape_as_prompt', 'Gus', 'Tara', 'set', 'Omar', 'prompt_overlap', 'stress_merged', 'Ethan', 'ratify', 'Ugo', 'Andrew', 'stratum', 'Vera', 'labels', 'Zoe', 'solution'}


In [ ]:

import json, re

JSON_OBJ = re.compile(r'\{[^{}]*\}')

def extract(reply, expected_keys):
    want = set(expected_keys)
    for m in JSON_OBJ.finditer(reply):
        try:
            obj = json.loads(m.group(0))
        except json.JSONDecodeError:
            continue
        if isinstance(obj, dict) and set(obj) == want and all(isinstance(v, bool) for v in obj.values()):
            return obj
    return None

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os

OUT_PATH  = "/content/drive/MyDrive/out_engaged_tests_all.json"
PROMPT    = SYSTEM_PROMPT
RERUN_IDS = None
results = json.load(open(OUT_PATH)) if os.path.exists(OUT_PATH) else []
done = {r["id"] for r in results}


for puzzle in file['puzzles']:
  names = list(puzzle['statements'].keys())
  roles = [puzzle['roles']['truth'], puzzle['roles']['liar']]

  for block in puzzle['examples']:

    if block["id"] in done: continue

    if RERUN_IDS is not None and block["id"] not in RERUN_IDS: continue

    blk = block['text']

    print(f'block text: {blk}\n')
    print('\n')

    messages = build_messages(names, blk)
    messages[0]["content"] = PROMPT
    messages[0]["content"] = messages[0]["content"].replace("<<TRUTH_ROLE>>", roles[0]).replace("<<LIAR_ROLE>>", roles[1])

    print('messages: ', messages)
    assert "<<" not in messages[0]["content"]


    THINKING = True   # flip for the other arm

    chat_prompt = pipe.tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
        enable_thinking=THINKING,
    )

    out = pipe(chat_prompt,
          max_new_tokens=4000 if THINKING else 300,
          do_sample=False)

    print("Qwen's Response:")

    reply = out[0]["generated_text"][len(chat_prompt):]
    raw = reply

    truncated = THINKING and "</think>" not in reply

    if "</think>" in reply:
        reply = reply.rsplit("</think>", 1)[1].strip()

    answer = None if truncated else extract(reply, names)
    print(block['id'], "truncated:", truncated)

    print(f'For block "{blk}",  the probe tests for link info: ')
    print(answer)

    print(f'labels for each role: "{block['labels']}')
    print('\n')

    results.append({"id": block["id"], "answer": answer,
                    "truncated": truncated, "raw": raw})
    with open(OUT_PATH, "w") as f:
        json.dump(results, f, indent=1)






[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=4000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


block text: Noah claims Ethan and Mia are both devils. But Mia is a saint, so Noah's statement is false. A false statement means Noah is a devil.



messages:  [{'role': 'system', 'content': 'Knights and Knaves. Each character makes one statement.\nCharacters with the role "saint" always speak the truth. Characters with the role "devil" always lie.\n\nYou are given the names of the characters and one block of someone\'s reasoning about the puzzle.\nYou are not given the puzzle\'s statements. Use only the text of this block.\nDo not try to solve the puzzle. Never check whether the block is right.\n\nFor EACH character listed, answer one question:\n\nDoes the block do BOTH of these for that character?\n  (A) say that THAT character\'s OWN statement is true or false\n  (B) connect that truth claim to something -- derive it from something the block states,\n      or draw something from it\n\nAbout (A)\n- Any plain wording that says the statement is true or false counts, for example "accura

[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qw

Qwen's Response:
pos-01 truncated: False
For block "Noah claims Ethan and Mia are both devils. But Mia is a saint, so Noah's statement is false. A false statement means Noah is a devil.",  the probe tests for link info: 
{'Ethan': False, 'Mia': False, 'Noah': True}
labels for each role: "[{'speaker': 'Noah', 'engaged': True, 'reason': "(A) 'Noah's statement is false'; (B) connected by 'so' and again by 'means'"}]


block text: Ethan is a devil, so his statement is false.



messages:  [{'role': 'system', 'content': 'Knights and Knaves. Each character makes one statement.\nCharacters with the role "saint" always speak the truth. Characters with the role "devil" always lie.\n\nYou are given the names of the characters and one block of someone\'s reasoning about the puzzle.\nYou are not given the puzzle\'s statements. Use only the text of this block.\nDo not try to solve the puzzle. Never check whether the block is right.\n\nFor EACH character listed, answer one question:\n\nDoes the bloc

[transformers] Both `max_new_tokens` (=4000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Qwen's Response:
R-01 truncated: False
For block "Therefore, Ethan must be a saint.",  the probe tests for link info: 
{'Joseph': False, 'Ethan': False}
labels for each role: "[{'speaker': 'Joseph', 'engaged': False, 'difficulty': 'easy', 'reason': 'no mention at all'}, {'speaker': 'Ethan', 'engaged': False, 'difficulty': 'hard', 'reason': 'role claim with a connective word, but no truth value for his own statement'}]


block text: Mia's statement is true.



messages:  [{'role': 'system', 'content': 'Knights and Knaves. Each character makes one statement.\nCharacters with the role "angel" always speak the truth. Characters with the role "devil" always lie.\n\nYou are given the names of the characters and one block of someone\'s reasoning about the puzzle.\nYou are not given the puzzle\'s statements. Use only the text of this block.\nDo not try to solve the puzzle. Never check whether the block is right.\n\nFor EACH character listed, answer one question:\n\nDoes the block do BOTH of th

[transformers] Both `max_new_tokens` (=4000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Qwen's Response:
R-02 truncated: False
For block "Mia's statement is true.",  the probe tests for link info: 
{'Mia': False, 'Ella': False}
labels for each role: "[{'speaker': 'Mia', 'engaged': False, 'difficulty': 'hard', 'reason': '(A) present but (B) fails: the truth claim is the whole block, nothing is joined. Same shape as B-04 in stress-v1'}, {'speaker': 'Ella', 'engaged': False, 'difficulty': 'easy', 'reason': 'no mention at all'}]


block text: Ella affirms Mia is an angel, aligning with Mia's statement.



messages:  [{'role': 'system', 'content': 'Knights and Knaves. Each character makes one statement.\nCharacters with the role "angel" always speak the truth. Characters with the role "devil" always lie.\n\nYou are given the names of the characters and one block of someone\'s reasoning about the puzzle.\nYou are not given the puzzle\'s statements. Use only the text of this block.\nDo not try to solve the puzzle. Never check whether the block is right.\n\nFor EACH character lis

[transformers] Both `max_new_tokens` (=4000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Qwen's Response:
R-03 truncated: False
For block "Ella affirms Mia is an angel, aligning with Mia's statement.",  the probe tests for link info: 
{'Mia': False, 'Ella': False}
labels for each role: "[{'speaker': 'Mia', 'engaged': False, 'difficulty': 'hard', 'reason': "'aligning with Mia's statement' compares two statements; it gives Mia's statement no truth value"}, {'speaker': 'Ella', 'engaged': False, 'difficulty': 'hard', 'reason': "'Ella affirms ...' repeats what Ella said; her statement is never called true or false"}]


block text: Luna being an oracle would mean her statement is true, implying she must be an oracle.



messages:  [{'role': 'system', 'content': 'Knights and Knaves. Each character makes one statement.\nCharacters with the role "oracle" always speak the truth. Characters with the role "trickster" always lie.\n\nYou are given the names of the characters and one block of someone\'s reasoning about the puzzle.\nYou are not given the puzzle\'s statements. Use only the

In [ ]:
from google.colab import runtime
runtime.unassign()